# Revenue Investigation

Load and prepare the data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/customers.csv')

# Executive Summary

Revenue declined approximately **20%** from the second half of 2023 to the second half of 2024 (from ~\$10.2M to ~\$8.2M). This is **driven by a drop in order volume** (about 26% fewer delivered orders), while average order value (AOV) actually increased. The decline is concentrated in **Active** and **VIP** customer segments, which contributed fewer orders in 2024 H2. We recommend launching retention and win-back campaigns for these segments and reinforcing seasonal promotions to restore order frequency.

# Analysis

Below we examine monthly revenue trends, order volume vs. average order value, and performance by customer segment.

In [ ]:
# Use delivered orders for revenue (exclude pending, returned, etc.)
delivered = orders[orders['status'] == 'delivered'].copy()

# Monthly revenue trend
monthly_revenue = delivered.groupby(delivered['order_date'].dt.to_period('M'))['total'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
monthly_revenue.plot(kind='bar', ax=ax, color='steelblue', edgecolor='navy', alpha=0.85)
ax.set_title('Monthly Revenue (Delivered Orders)')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Compare 2023 H2 vs 2024 H2: order volume and average order value
period_2023_h2 = delivered[(delivered['order_date'] >= '2023-07-01') & (delivered['order_date'] < '2024-01-01')]
period_2024_h2 = delivered[(delivered['order_date'] >= '2024-07-01') & (delivered['order_date'] < '2025-01-01')]

comparison = pd.DataFrame({
    '2023 H2': [
        period_2023_h2['total'].sum(),
        len(period_2023_h2),
        period_2023_h2['total'].mean()
    ],
    '2024 H2': [
        period_2024_h2['total'].sum(),
        len(period_2024_h2),
        period_2024_h2['total'].mean()
    ]
}, index=['Total Revenue ($)', 'Order Count', 'Avg Order Value ($)'])
comparison['Change %'] = (comparison['2024 H2'] - comparison['2023 H2']) / comparison['2023 H2'] * 100
comparison.round(2)

In [ ]:
# Revenue and order count by period (visual)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
periods = ['2023 H2', '2024 H2']
revenue_vals = [period_2023_h2['total'].sum(), period_2024_h2['total'].sum()]
order_counts = [len(period_2023_h2), len(period_2024_h2)]
ax1.bar(periods, revenue_vals, color=['steelblue', 'coral'], edgecolor='black')
ax1.set_ylabel('Revenue ($)')
ax1.set_title('Revenue: 2023 H2 vs 2024 H2')
ax2.bar(periods, order_counts, color=['steelblue', 'coral'], edgecolor='black')
ax2.set_ylabel('Number of Orders')
ax2.set_title('Order Volume: 2023 H2 vs 2024 H2')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by customer segment: 2023 H2 vs 2024 H2
merged = delivered.merge(customers[['customer_id', 'segment']], on='customer_id', how='left')
seg_2023 = merged[(merged['order_date'] >= '2023-07-01') & (merged['order_date'] < '2024-01-01')].groupby('segment')['total'].agg(['sum', 'count', 'mean'])
seg_2024 = merged[(merged['order_date'] >= '2024-07-01') & (merged['order_date'] < '2025-01-01')].groupby('segment')['total'].agg(['sum', 'count', 'mean'])
seg_2023.columns = ['Revenue_2023', 'Orders_2023', 'AOV_2023']
seg_2024.columns = ['Revenue_2024', 'Orders_2024', 'AOV_2024']
segment_comparison = seg_2023.join(seg_2024)
segment_comparison['Revenue_Change_%'] = (segment_comparison['Revenue_2024'] - segment_comparison['Revenue_2023']) / segment_comparison['Revenue_2023'] * 100
segment_comparison['Order_Count_Change_%'] = (segment_comparison['Orders_2024'] - segment_comparison['Orders_2023']) / segment_comparison['Orders_2023'] * 100
segment_comparison.round(2)

In [ ]:
# Segment revenue comparison chart
segments = segment_comparison.index
x = range(len(segments))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width/2 for i in x], segment_comparison['Revenue_2023'], width, label='2023 H2', color='steelblue')
ax.bar([i + width/2 for i in x], segment_comparison['Revenue_2024'], width, label='2024 H2', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(segments)
ax.set_ylabel('Revenue ($)')
ax.set_title('Revenue by Customer Segment: 2023 H2 vs 2024 H2')
ax.legend()
plt.tight_layout()
plt.show()

# Root Cause

The revenue drop is **not** caused by customers spending less per order—average order value (AOV) increased from 2023 H2 to 2024 H2. The main driver is **fewer orders**: delivered order count fell by about 26% year-over-year in the second half of the year.

Segment analysis shows the decline is concentrated in **Active** and **VIP** segments, which contribute the largest share of revenue. These segments placed significantly fewer orders in 2024 H2 compared with 2023 H2, even though their AOV held or improved. So the root cause is **lower order frequency** among our best customer segments—likely a mix of reduced engagement, fewer repeat purchases, and possibly some migration toward At Risk or Churned. Seasonal patterns (e.g. 2024 H2 monthly revenue softening vs earlier in 2024) support a trend rather than a one-off dip.

# Recommendations

1. **Launch retention and win-back campaigns for Active and VIP segments.** Use email and in-app messaging to remind lapsed or less-frequent buyers to return (e.g. “We miss you” offers, personalized product picks). Prioritize customers whose order frequency dropped in 2024.

2. **Increase order frequency with targeted promotions.** Run time-bound offers (e.g. free shipping, bundle discounts, or loyalty rewards) aimed at driving a second or third order within the half-year. Focus on segments that historically had higher order counts and have declined.

3. **Monitor and act on At Risk and Churned segments.** Invest in early win-back before customers fully churn, and track leading indicators (e.g. days since last order, basket size trends) so you can intervene with tailored offers before revenue is lost.